# Running inference on a remote machine

1. Install this repository (mujoco_test) on the cluster. (Tested with TACC)
2. Download the appropriate model checkpoint, into `pace/openpi/checkpoints` using the `download_lora_checkpoints` script.
    - You may have to fix norm stats manually
3. Implement your inference harness, following the format in `py_script/serving`. Some examples are provided for base pi05 inference (`pi05_action.py`) and some reasoning models (`pi05_plan_skill_action.py`)
    - At the lowest level, this involves placing a policy object factory function into the `MODEL_REGISTRY` global variable from `inference_common.py`. This factory should take no arguments, and return a `policy` with the following two methods:
        - `policy.initialize(observation: dict, task: str)`: Set the policy up to execute a given task, given the environment state as a dictionary in LIBERO format.
        - `policy.run_vla(observation: dict)`: Return a new action chunk, given the current observation.
    - Your policy object should keep track of any internal variables it needs (such as cached traces, or a counter for how many times inference should be called before replanning).
    - You can use `inference_common.register_model()` or `inference_common.get_model_info` as convenience functions, to sort out the paths to downloaded checkpoints and norm stats.
    - You should (at least) create a new python file in `serving`, that when imported, will register your model to `MODEL_REGISTRY` following the above conventions.
4. Modify the inference server run script (`serve_pi05.py`) to load and serve your registered model.
5. Install our i2rt robot driver fork to the robot computer. (this step is already done; it is under `workspace/hardware_setup/i2rt`
6. Source the normal virtualenv (`workspace/mujoco_test/mujoco_playground/.venv/bin/activate`). Then, go to `workspace/hardware_setup/i2rt/data_collection`:
    1. Start the follower robot: `python robot_recorder.py`
    2. Start the teacher robot (for resetting): `python data_collect_leader.py`
    3. Press the yellow button on the teacher robot to synchronize the follower robot to the teacher robot's position.
    4. Press the yellow button again, to desynchronize. VERY IMPORTANT! DO NOT RUN A POLICY WHILE THE ARMS ARE SYNCHRONIZED!
7. Reset cameras.
    - The easiest way to do this is to unplug both of them, and plug them in in order; start with the overhead camera (cam1, /dev/video4) and then do the wrist camera (cam2, /dev/video10).
8. Start the inference server on the compute cluster (requires sourcing env as usual):
    1. Run an interactive job (`idev` or `srun`). Note the node it starts on (ex. `c642-051`; visible in `squeue` usually)
    2. Create a SSH tunnel between the login node and the compute node: `ssh -NT -L 9898:localhost:9898 <node_name>`
        - 9898 is the default port for the inference server. Change this if you change the port in the inference server
    3. Create a SSH tunnel from the robot computer to the login node: `ssh -NT -L 9898:localhost:9898 <user>@<login_node_ip>`
10. Start running the rest of this notebook!

In [ ]:
%env MUJOCO_GL=osmesa
import functools
import json
import math
import os
import pathlib
import sys
import time

import cv2
import numpy as np
import mediapy as media

from openpi_client.websocket_client_policy import WebsocketClientPolicy

sys.path.append('../py_script')

In [ ]:
import cv2
from threading import Thread

class CameraReader:
    def __init__(self, camera_ids):
        self.cameras = [cv2.VideoCapture(i) for i in camera_ids]
        self.obs = None
        self.running = False
        self.run_thread = None

    def start(self):
        self.running = True
        self.run_thread = Thread(target=self._spin_thread)
        self.run_thread.start()

    def stop(self):
        self.running = False

    def _spin_thread(self):
        while self.running:
            obs = []
            for camera in self.cameras:
                obs.append(camera.read())
            self.obs = obs

    def get_images(self):
        results = self.obs # Pointer copy
        return [r[1] for r in results]

cameras = CameraReader([4, 10])
cameras.start()

In [ ]:
# This should show the overhead camera. If not, you should replug the cameras and reinitialize them by running the above cell.
import matplotlib.pyplot as plt
images = cameras.get_images()
plt.imshow(images[0][:, :, ::-1])
plt.show()

In [ ]:
DEFAULT_ROBOT_PORT=11333
import portal
from i2rt.robots.robot import Robot

class ClientRobot(Robot):
    """A simple client for a follower robot."""

    def __init__(self, port: int = DEFAULT_ROBOT_PORT, host: str = "127.0.0.1"):
        self._client = portal.Client(f"{host}:{port}")

    def get_joint_pos(self) -> np.ndarray:
        """Get the current joint position of the follower obot.

        Returns:
            T: The current joint position of the follower robot: 6 dof, then gripper (0-1).
        """
        return self._client.get_joint_pos().result()

    def command_joint_pos(self, joint_pos: np.ndarray) -> None:
        """Command the leader robot to the given joint position.

        Args:
            joint_pos (T): The joint position to command the follower robot to: 6 dof, then gripper (0-1).
        """
        self._client.command_joint_pos(joint_pos)

    def get_tcp_pose(self) -> np.ndarray:
        """Get the current EE state of the follower obot.

        Returns:
            T: 4x4 matrix of end effector pose, relative to robot base frame.
        """
        return self._client.get_tcp_pose().result()

    def command_tcp_pose(self, tcp_pose: np.ndarray, gripper: float) -> None:
        """Command the leader robot to the given EE pose and gripper state.

        Args:
            tcp_pose (4x4 mat): TCP pose in robot base coordinates to move to.
            gripper: 0-1 gripper command (i think 1 is open, 0 is closed)
        """
        self._client.command_tcp_pose(tcp_pose, gripper)

    def get_observations():
        pass
    def num_dofs():
        pass

# Connect to a robot server running locally. (robot_recorder.py)
robot = ClientRobot()

In [ ]:
# Helper function for making the standard observation (image, wrist_image, state)
import mujoco
def flatten_pose(pose_mat):
    """
    Convert a 4x4 pose matrix to a flattened 7-d vector: (quat, pos)
    """
    q = np.zeros(4)
    mujoco.mju_mat2Quat(q, pose_mat[:3, :3].flatten())
    return np.concatenate([q, pose_mat[:3, 3]])

def make_obs(mode="ee"):
    images = cameras.get_images()
    if mode == "joint":
        state = robot.get_joint_pos()
    else:
        state = np.concatenate([
            flatten_pose(robot.get_tcp_pose()),
            [robot.get_joint_pos()[-1]]
        ])
    return {
        "image": images[0][:, :, ::-1],
        "wrist_image": images[1][:, :, ::-1],
        "state": state
    }

# Connect to a policy server running on localhost:9898. Usually this is ssh tunnelled to the execution machine
policy = WebsocketClientPolicy(port=9898)

In [ ]:
# Set the task, and initialize the policy.
# If any of the subsequent cells crash you may need to reinitialize the policy client (rerun the above cell); common if you interrupt a running cell.

# task = "put the green bell pepper on the blue plate"
task = "pick up the orange plate"
policy.initialize(None, task)

t0 = time.monotonic()
vla_output = policy.infer(make_obs())
t1 = time.monotonic()
print(t1 - t0, "elapsed")
actions = vla_output['actions']
print(actions)

In [ ]:
# This is legacy, since my previous trained model did not do gripper limits correctly.
# For the new datasets, you should use [0, 1]
#gripper_limits = [6.355838298797607, 1.1629282236099243]
gripper_limits = [0, 1]
gripper_range = gripper_limits[1] - gripper_limits[0]


def joint_command(obs, action):
    # Commands are deltas, except gripper which is absolute
    state = obs['state']
    command = state
    command[:-1] += action[:-1]
    command[-1] = (action[-1] - gripper_limits[0]) / gripper_range
    robot.command_joint_pos(command)

from motionlib so3
def unflatten_pose6(pose_6d):
    pose = np.eye(4, dtype=np.float32)
    pose[:3, :3] = so3.matrix(so3.from_moment(pose_6d[:3]))
    pose[:3, 3] = pose_6d[3:]
    return pose
def unflatten_pose7(pose_7d):
    pose = np.eye(4, dtype=np.float32)
    rot_flat = np.empty(9, dtype=np.float64)
    mujoco.mju_quat2Mat(rot_flat, pose_7d[:4])
    pose[:3, :3] = rot_flat.reshape((3, 3))
    pose[:3, 3] = pose_7d[4:]
    return pose

def EE_command(obs, action):
    current_pose = unflatten_pose7(obs['state'][:-1])
    error_6d = action[:-1]   # This is an error between the target and the current state, but in the world frame
    delta_pose = unflatten_pose6(error_6d)
    # So we reconstruct it by applying the rotation and translation separately
    reconstruct_command = np.eye(4)
    reconstruct_command[:3, :3] = delta_pose[:3, :3] @ current_pose[:3, :3]
    reconstruct_command[:3, 3] = delta_pose[:3, 3] + current_pose[:3, 3]
    robot.command_tcp_pose(reconstruct_command, action[-1])

# TODO: make this whole pipeline less jank
# Run for 30 iterations and see what it does.
video_images = []
wrist_video_images = []
for i in range(30):
    vla_output = policy.infer(make_obs())
    actions = vla_output['actions']
    
    for action in actions:
        obs = make_obs()
        # NOTE: change this to joint or EE command
        joint_command(obs, action)
        video_images.append(obs['image'])
        wrist_video_images.append(obs['wrist_image'])
        # This should probably be changed to make the overall loop time 1/30 s (measure and sleep the remainder)
        # it is critical that the policy runs at the same frequency due to robot / controller dynamics being learned.
        # Deviating too far will cause the policy to implode
        time.sleep(1/30)

In [ ]:
import mediapy
mediapy.write_video("bell_pepper_wrist.mp4", video_images)
mediapy.write_video("bell_pepper_long_wrist.mp4", wrist_video_images)